In [19]:
import pandas as pd

train = pd.read_parquet("data/processed/notebook3_train.parquet")
val = pd.read_parquet("data/processed/notebook3_val.parquet")
test = pd.read_parquet("data/processed/notebook3_test.parquet")

print("train:", train.shape)
print("val:", val.shape)
print("test:", test.shape)

train: (67529, 18)
val: (14470, 18)
test: (14471, 18)


# **Time features**

In [20]:
# أعمدة لازم تُستبعد قطعيًا
# (Data Leakage)
leakage_cols = [
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_duration",
]

#  ID
id_cols = ["order_id", "customer_id", "customer_unique_id"]

# Candidate Features Based on Notebook 4
numeric_features = [
    "total_price",
    "total_freight",
    "n_items",
]  # استبعدنا total_payment لتشابهه مع total_price
categorical_features = ["customer_state"]
time_features_source = "order_purchase_timestamp"  # هنبني منه purchase_month بعدين

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Time feature source:", time_features_source)

Numeric features: ['total_price', 'total_freight', 'n_items']
Categorical features: ['customer_state']
Time feature source: order_purchase_timestamp


In [21]:
def add_time_features(df):
    df = df.copy()
    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["purchase_dayofweek"] = df["order_purchase_timestamp"].dt.dayofweek
    df["is_holiday_season"] = (
        df["order_purchase_timestamp"].dt.month.isin([11, 12]).astype(int)
    )
    return df


train = add_time_features(train)
val = add_time_features(val)
test = add_time_features(test)

train[
    [
        "order_purchase_timestamp",
        "purchase_month",
        "purchase_dayofweek",
        "is_holiday_season",
    ]
].head()

,order_purchase_timestamp,purchase_month,purchase_dayofweek,is_holiday_season
0,2016-09-15 12:16:38,9,3,0
1,2016-10-03 09:44:50,10,0,0
2,2016-10-03 16:56:50,10,0,0
3,2016-10-03 21:13:36,10,0,0
4,2016-10-03 22:06:03,10,0,0


# **Categorical features**

In [22]:
state_counts = train["customer_state"].value_counts()
rare_states = state_counts[state_counts < 100].index.tolist()

print("the rare states:", rare_states)
print("their count:", len(rare_states))

the rare states: ['AC', 'AP', 'RR']
their count: 3


In [23]:
import numpy as np


def group_states(df, rare_list):
    df = df.copy()
    df["customer_state_grouped"] = np.where(
        df["customer_state"].isin(rare_list), "Other", df["customer_state"]
    )
    return df


train = group_states(train, rare_states)
val = group_states(val, rare_states)
test = group_states(test, rare_states)

train["customer_state_grouped"].value_counts().tail(10)

customer_state_grouped
MS       502
PB       370
RN       349
PI       348
AL       298
SE       246
TO       197
RO       185
Other    146
AM       105
Name: count, dtype: int64

In [24]:
from sklearn.preprocessing import OneHotEncoder

enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
enc.fit(train[["customer_state_grouped"]])

train_enc = enc.transform(train[["customer_state_grouped"]])
print("shape:", train_enc.shape)
print("cols:", enc.get_feature_names_out())
# handle_unknown="ignore" → حماية من فئات غريبة تظهر بـ val/test ما شافها الـ encoder
# sparse_output=False → سهولة التعامل مع النتيجة كجدول عادي بدل تنسيق خاص معقد

shape: (67529, 25)
cols: ['customer_state_grouped_AL' 'customer_state_grouped_AM'
 'customer_state_grouped_BA' 'customer_state_grouped_CE'
 'customer_state_grouped_DF' 'customer_state_grouped_ES'
 'customer_state_grouped_GO' 'customer_state_grouped_MA'
 'customer_state_grouped_MG' 'customer_state_grouped_MS'
 'customer_state_grouped_MT' 'customer_state_grouped_Other'
 'customer_state_grouped_PA' 'customer_state_grouped_PB'
 'customer_state_grouped_PE' 'customer_state_grouped_PI'
 'customer_state_grouped_PR' 'customer_state_grouped_RJ'
 'customer_state_grouped_RN' 'customer_state_grouped_RO'
 'customer_state_grouped_RS' 'customer_state_grouped_SC'
 'customer_state_grouped_SE' 'customer_state_grouped_SP'
 'customer_state_grouped_TO']


In [25]:
val_enc = enc.transform(val[["customer_state_grouped"]])
test_enc = enc.transform(test[["customer_state_grouped"]])

print("val shape:", val_enc.shape)
print("test shape:", test_enc.shape)
"""استخدمنا enc.transform(...) بس — مش enc.fit_transform(...). هاد الفرق يلي يضمن عدم تسريب أي معلومة من val/test لعملية "التعلم" — 
الـ encoder أصلاً ما شاف بيانات val/test إطلاقًا وقت التدريب، وبس عم "يطبق" القرار يلي اتعلمه من train عليهم.
"""

val shape: (14470, 25)
test shape: (14471, 25)


'استخدمنا enc.transform(...) بس — مش enc.fit_transform(...). هاد الفرق يلي يضمن عدم تسريب أي معلومة من val/test لعملية "التعلم" — \nالـ encoder أصلاً ما شاف بيانات val/test إطلاقًا وقت التدريب، وبس عم "يطبق" القرار يلي اتعلمه من train عليهم.\n'

### One-Hot Encoding لـ `customer_state_grouped`

- جمعنا الولايات النادرة (RR, AP, AC — أقل من 100 صف بالـ train) بفئة وحدة اسمها `"Other"` (146 صف)، بدل ما نخلي كل ولاية لحالها.

- درّبنا `OneHotEncoder` على `train` بس (`handle_unknown="ignore"` عشان نحمي حالنا من أي فئات جديدة ممكن تظهر بـ `val/test`).

- النتيجة: طلع **25 عمود ثنائي (0/1)** لكل من `train/val/test` — ونفس عدد الأعمدة بالضبط بالثلاثة، رغم إنو الـ encoder تدرب على `train` بس، وهاد بأكد إنو استخدمنا `transform` مش `fit_transform` على `val/test`.

# **Seller features**

In [26]:
from sqlalchemy import create_engine

DB_USER = "postgres"
DB_PASSWORD = "mysecretpassword"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "olist"

engine = create_engine(
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

order_items = pd.read_sql("SELECT order_id, seller_id FROM order_items", engine)
print(order_items.shape)
order_items.head()

(112650, 2)


,order_id,seller_id
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87


In [27]:
n_sellers = order_items.groupby("order_id")["seller_id"].nunique().reset_index()
n_sellers.columns = ["order_id", "n_sellers"]

print(n_sellers.shape)
n_sellers["n_sellers"].value_counts().sort_index()

(98666, 2)


n_sellers
1    97388
2     1219
3       54
4        3
5        2
Name: count, dtype: int64

In [28]:
train = train.merge(n_sellers, on="order_id", how="left")
val = val.merge(n_sellers, on="order_id", how="left")
test = test.merge(n_sellers, on="order_id", how="left")

train["n_sellers"].isnull().sum()

np.int64(0)

In [29]:
train.groupby("n_sellers")["label"].value_counts(normalize=True).unstack()

label,Late,On-time
n_sellers,,
1,0.091082,0.908918
2,0.021563,0.978437
3,NaN,1.000000
4,NaN,1.000000
5,NaN,1.000000


## Seller Feature (n_sellers)

**السبب:**
بـ Notebook 4 كنا وثقنا إنو بيانات البائع (sellers) محدودية معروفة مؤجلة عمدًا لمرحلة Feature Engineering — يعني هون بالضبط بـ Notebook 5.

**القرار:**
ندمجها جوا Notebook 5 مباشرة (بدون الرجوع لـ Notebook 1 وإعادة تشغيل السلسلة كاملة)، لأنها feature إضافي مش جزء من تعريف الـ label أو الـ split.

**استخدام order_id:**
استخدمناه بس كـ **"مفتاح ربط" (merge key)** لدمج n_sellers مع الجدول، مش كـ feature يدخل الموديل — نفس الطريقة يلي استخدمناها بكل عمليات الدمج بـ Notebook 1. دوره ينتهي بعد الـ merge؛ العمود الفعلي يلي بيدخل الموديل هو **n_sellers**.

**الخطوات:**
- قرأنا `order_id` و`seller_id` من `order_items`
- حسبنا `n_sellers` (عدد البائعين الفريدين لكل طلبية عبر `nunique`)
- دمجناه مع `train`/`val`/`test`

**النتيجة:**
- **التوزيع:** 98.7% من الطلبيات عندها بائع واحد بس (97,388 من أصل ~98,666)
- **العلاقة مع label:** نسبة Late تنخفض مع زيادة عدد البائعين (9.11% → 2.16% → 0%) — عكس الفرضية النظرية، لكن مبني على عينات صغيرة جدًا (2-54 صف للفئات 2-5)

**القرار النهائي:**
استبعاد `n_sellers` من الـ features النهائية بسبب:
1. Variance شبه معدوم
2. العلاقة غير الموثوقة إحصائيًا مع label

# **Numeric features**

In [30]:

log_cols = ["total_price", "total_freight", "n_items"]
for col in log_cols:
    # حلقة بتلف على الأعمدة الثلاثة وحدة وحدة (col ياخد كل اسم بالتتابع)
    # y=log(1+x)
    train[f"{col}_log"] = np.log1p(train[col])
    val[f"{col}_log"] = np.log1p(val[col])
    test[f"{col}_log"] = np.log1p(test[col])

train[[f"{c}_log" for c in log_cols]].describe()
#  عشان نتأكد إنو التحويل صار صح وشكل التوزيع تغيّر

,total_price_log,total_freight_log,n_items_log
count,67529.000000,67529.000000,67529.000000
mean,4.452711,2.984244,0.743936
std,0.916047,0.505623,0.167024
min,1.190888,0.000000,0.693147
25%,3.848018,2.714695,0.693147
50%,4.454347,2.878637,0.693147
75%,5.016617,3.206803,0.693147
max,9.506065,6.911040,3.091042


### نتيجة Log-Transformation على الأعمدة الرقمية

- `total_price_log`: mean=4.45, std=0.92, مدى (0-9.51) بدل (2.29-13,440)

- `total_freight_log`: mean=2.98, std=0.51, مدى (0-6.91) بدل (0-1,002.29)

- `n_items_log`: mean=0.74, std=0.17, مدى (0.69-3.09) بدل (1-21)

التحويل نجح في تقليص القيم المتطرفة بشكل كبير جدًا — القيم اللي كانت بالآلاف (13,440، 1,002) صارت بمدى صغير (أقل من 10)، مما يقلل تأثير الـ outliers ويقرّب التوزيع من التناظر.

### ملاحظة

`n_items_log` عندها نفس القيمة (0.693) بربيعي 25%/50%/75% — منطقي لأن الوسيط الأصلي لـ `n_items` كان 1 (أغلب الطلبيات منتج واحد)، و`log1p(1) = 0.693` بالضبط.

In [31]:
from sklearn.preprocessing import StandardScaler

scale_cols = ["total_price_log", "total_freight_log", "n_items_log"]

scaler = StandardScaler()
scaler.fit(train[scale_cols])

train_scaled = scaler.transform(train[scale_cols])
val_scaled = scaler.transform(val[scale_cols])
test_scaled = scaler.transform(test[scale_cols])

print("train shape:", train_scaled.shape)

train shape: (67529, 3)


### StandardScaler على الأعمدة الرقمية (بعد log-transform)

- تم تدريب `StandardScaler` على train فقط (mean وstd محسوبين من train)، ثم تطبيقه (transform) على val وtest.

- الأعمدة المُحوّلة: `total_price_log`, `total_freight_log`, `n_items_log`

- الترتيب: log-transform أولاً (يعالج شكل التوزيع/skew)، ثم scaling (يعالج المقياس/الحجم النسبي بين الأعمدة).

- النتيجة: train `(67529, 3)` — الأعمدة الآن بمتوسط≈0 وانحراف معياري≈1، جاهزة لموديلات حساسة للمقياس مثل Logistic Regression.

# **Missing values handling**

In [32]:
final_features = [
    "total_price_log",
    "total_freight_log",
    "n_items_log",
    "purchase_month",
    "purchase_dayofweek",
    "is_holiday_season",
]

train[final_features].isnull().sum()

total_price_log       0
total_freight_log     0
n_items_log           0
purchase_month        0
purchase_dayofweek    0
is_holiday_season     0
dtype: int64

### معالجة القيم الناقصة — النتيجة: لا حاجة لـ imputation

- فحصنا القيم الناقصة على الأعمدة الستة النهائية (`final_features`) وطلعت 0 قيم ناقصة بكل واحد منهم.

- السبب: الأعمدة الأصلية اللي بُنيت منها (`total_price`, `total_freight`, `n_items`, `order_purchase_timestamp`) لم تحتوِ على أي قيم ناقصة أصلاً (تأكيد من فحوصات Notebook 4).

- الأعمدة اللي كان فيها نقص فعلي (`order_approved_at`: 14 صف، `total_payment` و`n_payments`: صف واحد لكل) استُبعدت مسبقًا من قائمة الـ features لأسباب أخرى (multicollinearity، أو لأنها ليست features مباشرة).

- القرار: لا حاجة لخطوة imputation في هذا المشروع — قسم "Missing values handling" موثق كخطوة تم فحصها والتأكد من عدم الحاجة إليها، وليست خطوة تم تجاهلها.

# **The final stage**

In [33]:
import pandas as pd


def build_final_table(df, encoded_array, encoder):
    encoded_df = pd.DataFrame(
        encoded_array, columns=encoder.get_feature_names_out(), index=df.index
    )
    numeric_part = df[
        [
            "total_price_log",
            "total_freight_log",
            "n_items_log",
            "purchase_month",
            "purchase_dayofweek",
            "is_holiday_season",
        ]
    ]
    label_part = df[["label"]]
    final_df = pd.concat([numeric_part, encoded_df, label_part], axis=1)
    return final_df


train_final = build_final_table(train, train_enc, enc)
val_final = build_final_table(val, val_enc, enc)
test_final = build_final_table(test, test_enc, enc)

print("train_final:", train_final.shape)
print("val_final:", val_final.shape)
print("test_final:", test_final.shape)
train_final.head()

train_final: (67529, 32)
val_final: (14470, 32)
test_final: (14471, 32)


,total_price_log,total_freight_log,n_items_log,purchase_month,purchase_dayofweek,is_holiday_season,customer_state_grouped_AL,customer_state_grouped_AM,customer_state_grouped_BA,customer_state_grouped_CE,...,customer_state_grouped_PR,customer_state_grouped_RJ,customer_state_grouped_RN,customer_state_grouped_RO,customer_state_grouped_RS,customer_state_grouped_SC,customer_state_grouped_SE,customer_state_grouped_SP,customer_state_grouped_TO,label
0,4.912434,2.250239,1.386294,9,3,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,Late
1,3.430756,2.806990,0.693147,10,0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,On-time
2,3.131137,2.900872,0.693147,10,0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,On-time
3,3.624074,2.903617,0.693147,10,0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,On-time
4,4.794964,2.678278,0.693147,10,0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,On-time


In [34]:
import os

import joblib

os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/models", exist_ok=True)

# 1. Save the final feature tables
train_final.to_parquet("data/processed/notebook5_train_features.parquet", index=False)
val_final.to_parquet("data/processed/notebook5_val_features.parquet", index=False)
test_final.to_parquet("data/processed/notebook5_test_features.parquet", index=False)

# 2. Save the fitted transformers
joblib.dump(enc, "data/models/notebook5_state_encoder.pkl")
joblib.dump(scaler, "data/models/notebook5_scaler.pkl")
joblib.dump(rare_states, "data/models/notebook5_rare_states.pkl")

# 3. Save the final feature name list
feature_names = [c for c in train_final.columns if c != "label"]
joblib.dump(feature_names, "data/models/notebook5_feature_names.pkl")

print("All artifacts saved successfully")
print("Number of final features:", len(feature_names))

All artifacts saved successfully
Number of final features: 31
